# M1–M3 Final Joint Intent + Slot Ensemble

### Three joint models

| Model | Architecture | Original checkpoint |
|---|---|---|
| **M1** | 300D GloVe → BiLSTM → attention → intent + BIOES-CRF slot heads | `M1_best.pt` |
| **M2** | M1 + residual Character CNN features → intent + BIOES-CRF slot heads | `M2_best.pt` (CharacterCNN) |
| **M3** | Frozen GloVe + learned BPE features → 2-layer BiLSTM → intent + BIOES-CRF slot heads | `M3_best.pt` |

M1 training includes its original preliminary initialization phase. M2 training uses M1's compatible parameters as a warm start. M3 trains independently.

### Recommended workflow

1. Edit **Section 1** to choose `train` or `load` for **each** model.
2. Place `en-US.jsonl` alongside the notebook; use `glove.6B.300d.txt` only when required for training.
3. Run cells top to bottom. The notebook reports model-by-model training timestamps, epochs, validation metrics and total parameters.
4. It computes a 3-way ensemble by averaging intent probabilities and CRF scores, with the original **1/3 weight per model**.
5. Keep `RUN_TEST_ONCE = False` until the final configuration is frozen.



## 1. Configuration — choose `train` or `load` per model

Change only this section for ordinary runs. A model in `"load"` mode uses its historical saved checkpoint; a model in `"train"` mode trains and saves its own best checkpoint into the output directory.

When **M1 trains**, its initialization stage can either be newly trained or initialized from the saved base checkpoint. **M2 starts from M1 when training**, so you can load M1 while training M2. Training M3 requires GloVe and builds its own BPE tokenizer. Loading M3 requires the matching saved tokenizer JSON.


In [1]:
from pathlib import Path

# ---------------------------------------------------------------------
# Input data and output files
# ---------------------------------------------------------------------
DATA_PATH = Path("en-US.jsonl")
GLOVE_PATH = Path("glove.6B.300d.txt")
OUTPUT_DIR = Path("final_M1_M2_M3_outputs")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# Always load the tokenizer saved with the corrected M3 checkpoint.
BPE_LOAD_PATH = OUTPUT_DIR / "M3_bpe_tokenizer.json"

# ---------------------------------------------------------------------
# Train/load settings. Change these independently for M1, M2 and M3.
# ---------------------------------------------------------------------
MODE = {
    "M1": "train",  # "train" or "load"
    "M2": "train",
    "M3": "train",
}

# If M1 is training, choose how its initial BiLSTM-CRF phase is obtained.
# "train": train the base phase from scratch using GloVe.
# "load":  start from the original best base-phase checkpoint.
M1_INIT_MODE = "train"  # "train" or "load"

LOAD_PATH = {
    "M1_init": OUTPUT_DIR / "M1_initialization_best.pt",
    "M1": OUTPUT_DIR / "M1_best.pt",
    "M2": OUTPUT_DIR / "M2_best.pt",
    "M3": OUTPUT_DIR / "M3_best.pt",
}

# --------------------------------------------------3------------------
# Reproducibility and limits
# ---------------------------------------------------------------------
SEED = 42
PARAMETER_LIMIT = 20_000_000
RUN_TEST_ONCE = False  # Run test only after model/weights are finalized.

# Keep these settings unchanged for comparison with prior experiments.
BATCH_SIZE = 32
M1_BATCH_SIZE = 64
WEIGHT_DECAY = 1e-4

M1_INIT_EPOCHS = 12
M1_EPOCHS = 20
M2_EPOCHS = 20
M3_EPOCHS = 20

M1_LR = 2e-4
M2_LR = 2e-4
M3_LR = 3e-4

M1_PATIENCE = 6
M2_PATIENCE = 5
M3_PATIENCE = 5

BPE_VOCAB_SIZE = 2_000
SUBWORD_DIM = 128
M3_HIDDEN_SIZE = 256
M3_LAYERS = 2
M3_DROPOUT = 0.3

# Three models have equal weights for BOTH intent and slot predictions.
FUSION_WEIGHTS = {
    "M1": 1 / 3,
    "M2": 1 / 3,
    "M3": 1 / 3,
}

# ---------------------------------------------------------------------
# Check the requested files before starting training or evaluation.
# ---------------------------------------------------------------------
for model_name, selected_mode in MODE.items():
    if selected_mode not in {"train", "load"}:
        raise ValueError(f"MODE[{model_name!r}] must be 'train' or 'load'")

    if selected_mode == "load" and not LOAD_PATH[model_name].is_file():
        raise FileNotFoundError(f"Missing {model_name} checkpoint: {LOAD_PATH[model_name]}")

if M1_INIT_MODE not in {"train", "load"}:
    raise ValueError("M1_INIT_MODE must be 'train' or 'load'")

if MODE["M1"] == "train" and M1_INIT_MODE == "load":
    if not LOAD_PATH["M1_init"].is_file():
        raise FileNotFoundError(LOAD_PATH["M1_init"])

if MODE["M3"] == "load" and not BPE_LOAD_PATH.is_file():
    raise FileNotFoundError(f"Missing M3 tokenizer: {BPE_LOAD_PATH}")

print("Selected modes:", MODE)
print("Output directory:", OUTPUT_DIR.resolve())


Selected modes: {'M1': 'train', 'M2': 'train', 'M3': 'train'}
Output directory: C:\Users\cianm\Desktop\School Projects\SC4002 - Natural Language Processing\final_version\final_M1_M2_M3_outputs_corrected


## 2. Imports, random seeds, and Singapore-time logging

Timing is tracked for every epoch, training run, checkpoint load, and ensemble evaluation. A CUDA synchronization is used around clocks so GPU computations are included in the elapsed time.


In [2]:
import gc
import json
import random
import re
import time
from collections import Counter
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, Dataset
from IPython.display import display
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

SINGAPORE_TIMEZONE = timezone(timedelta(hours=8))
TIMING_ROWS = []


def now_sgt():
    """Return the current Singapore local timestamp (UTC+08:00)."""
    return datetime.now(SINGAPORE_TIMEZONE).isoformat(timespec="seconds")


def synchronize():
    """Wait for queued CUDA work so GPU elapsed time is measured correctly."""
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def start_clock(label):
    """Record the start timestamp and a precise elapsed-time counter."""
    synchronize()
    started_at = now_sgt()
    started_counter = time.perf_counter()
    print(f"[{started_at}] START {label}", flush=True)
    return started_at, started_counter


def stop_clock(label, clock, action="train"):
    """Print and store a time log row; return elapsed seconds."""
    synchronize()
    finished_at = now_sgt()
    elapsed_seconds = time.perf_counter() - clock[1]

    print(
        f"[{finished_at}] END {label} | "
        f"elapsed {timedelta(seconds=round(elapsed_seconds))}",
        flush=True,
    )

    TIMING_ROWS.append({
        "component": label,
        "action": action,
        "started_at_sgt": clock[0],
        "ended_at_sgt": finished_at,
        "elapsed_seconds": round(elapsed_seconds, 2),
    })
    return elapsed_seconds


def seed_everything(seed=42):
    """Initialize Python, NumPy, CPU PyTorch and CUDA RNGs."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_file_path = DATA_PATH
print(f"[{now_sgt()}] Device: {device}; PyTorch: {torch.__version__}")


[2026-11-11T17:15:37+08:00] Device: cuda; PyTorch: 2.11.0+cu128


## 3. MASSIVE data, direct BIOES labeling, and exact-span slot metrics

MASSIVE's `annot_utt` field contains bracketed slot annotations. We recover the original text and character offsets, then create **BIOES** labels directly. All label and word vocabularies are built using the training split only. Dev is used for selection; test remains untouched.


In [3]:
if not data_file_path.exists():
    raise FileNotFoundError(
        f"Cannot find {data_file_path}. Put en-US.jsonl beside this notebook."
    )

with data_file_path.open("r", encoding="utf-8") as data_file:
    all_records = [json.loads(line) for line in data_file if line.strip()]

training_records = [record for record in all_records if record["partition"] == "train"]
validation_records = [record for record in all_records if record["partition"] == "dev"]
test_records = [record for record in all_records if record["partition"] == "test"]

print("Training examples:", len(training_records))
print("Validation examples:", len(validation_records))
print("Test examples:", len(test_records))

assert len(training_records) == 11514
assert len(validation_records) == 2033
assert len(test_records) == 2974
assert len(all_records) == 16521
assert all(record["locale"] == "en-US" for record in all_records)

print("Example input:", training_records[0]["utt"])
print("Example intent:", training_records[0]["intent"])
print("Example annotation:", training_records[0]["annot_utt"])

Training examples: 11514
Validation examples: 2033
Test examples: 2974
Example input: wake me up at nine am on friday
Example intent: alarm_set
Example annotation: wake me up at [time : nine am] on [date : friday]


In [4]:
# MASSIVE represents slots using square-bracket annotations.
# Example: "go to [place_name : New York]".
SLOT_ANNOTATION_PATTERN = re.compile(r"\[([^:\]]+)\s*:\s*([^\]]+)\]")
WORD_PATTERN = re.compile(r"\S+")


def parse_annotated_utterance(annotated_utterance):
    """Return the plain utterance and each annotated slot's character offsets.

    Returned slots are tuples: (slot_type, start_character, end_character).
    The end index is exclusive, just like Python string slicing.
    """
    original_text_parts = []
    slot_spans = []
    current_character_position = 0
    previous_annotation_end = 0

    for annotation in SLOT_ANNOTATION_PATTERN.finditer(annotated_utterance):
        # Copy ordinary text that appears before the next slot.
        text_before_slot = annotated_utterance[
            previous_annotation_end:annotation.start()
        ]
        original_text_parts.append(text_before_slot)
        current_character_position += len(text_before_slot)

        # Record the slot's location in the reconstructed plain text.
        slot_type = annotation.group(1).strip()
        slot_text = annotation.group(2)
        slot_start = current_character_position
        slot_end = slot_start + len(slot_text)

        original_text_parts.append(slot_text)
        slot_spans.append((slot_type, slot_start, slot_end))
        current_character_position = slot_end
        previous_annotation_end = annotation.end()

    # Include any ordinary text following the final slot.
    original_text_parts.append(annotated_utterance[previous_annotation_end:])
    original_text = "".join(original_text_parts)

    return original_text, slot_spans


def tokenize_with_offsets(utterance):
    """Split on non-whitespace words while preserving exact character positions."""
    tokens_with_offsets = []

    for token_match in WORD_PATTERN.finditer(utterance):
        token = token_match.group()
        start_character = token_match.start()
        end_character = token_match.end()
        tokens_with_offsets.append((token, start_character, end_character))

    return tokens_with_offsets


def convert_spans_to_bioes(token_offsets, slot_spans):
    """Convert character-level slot spans into word-level BIOES labels.

    BIOES meanings:
    - B: Beginning of a multiword slot
    - I: Inside a multiword slot
    - O: Outside any slot
    - E: End of a multiword slot
    - S: A single-word slot
    """
    labels = ["O"] * len(token_offsets)

    for slot_type, slot_start, slot_end in slot_spans:
        overlapping_token_indices = []

        for token_index, (_, token_start, token_end) in enumerate(token_offsets):
            overlaps_slot = token_start < slot_end and token_end > slot_start
            if overlaps_slot:
                overlapping_token_indices.append(token_index)

        if not overlapping_token_indices:
            raise ValueError(
                f"No token found for slot {slot_type} at {slot_start}:{slot_end}"
            )

        first_token_index = overlapping_token_indices[0]
        last_token_index = overlapping_token_indices[-1]

        # Exact boundaries are necessary for exact-span slot evaluation.
        starts_at_token_boundary = token_offsets[first_token_index][1] == slot_start
        ends_at_token_boundary = token_offsets[last_token_index][2] == slot_end

        if not (starts_at_token_boundary and ends_at_token_boundary):
            raise ValueError(f"Slot not aligned with token boundaries: {slot_type}")

        if any(labels[index] != "O" for index in overlapping_token_indices):
            raise ValueError("Overlapping slot annotations are not supported")

        if len(overlapping_token_indices) == 1:
            labels[first_token_index] = f"S-{slot_type}"
        else:
            labels[first_token_index] = f"B-{slot_type}"
            for token_index in overlapping_token_indices[1:-1]:
                labels[token_index] = f"I-{slot_type}"
            labels[last_token_index] = f"E-{slot_type}"

    return labels


# Show one concrete example before training.
sample_record = training_records[0]
sample_text, sample_slot_spans = parse_annotated_utterance(sample_record["annot_utt"])
sample_tokens = tokenize_with_offsets(sample_text)
sample_bioes_labels = convert_spans_to_bioes(sample_tokens, sample_slot_spans)

print("Original sentence:", sample_text)
print("Tokens:", [token for token, _, _ in sample_tokens])
print("BIOES labels:", sample_bioes_labels)
print(
    "Gold slots:",
    [
        (slot_type, sample_text[start:end])
        for slot_type, start, end in sample_slot_spans
    ],
)

Original sentence: wake me up at nine am on friday
Tokens: ['wake', 'me', 'up', 'at', 'nine', 'am', 'on', 'friday']
BIOES labels: ['O', 'O', 'O', 'O', 'B-time', 'E-time', 'O', 'S-date']
Gold slots: [('time', 'nine am'), ('date', 'friday')]


In [5]:
dataset_partitions = {
    "train": training_records,
    "dev": validation_records,
    "test": test_records,
}

for partition_name, records in dataset_partitions.items():
    for record in records:
        reconstructed_text, slot_spans = parse_annotated_utterance(record["annot_utt"])
        assert reconstructed_text == record["utt"], (
            f"Text mismatch: {partition_name}, record {record['id']}"
        )
        token_offsets = tokenize_with_offsets(record["utt"])
        convert_spans_to_bioes(token_offsets, slot_spans)

print("Every annotation was reconstructed and aligned successfully.")

Every annotation was reconstructed and aligned successfully.


In [6]:
# Build all classifications from the TRAINING split only.
# In particular, do not create new labels from validation/test examples.
word_frequencies = Counter()
known_intent_labels = set()
known_slot_types = set()

for record in training_records:
    token_offsets = tokenize_with_offsets(record["utt"])

    for token, token_start, token_end in token_offsets:
        word_frequencies[token.lower()] += 1

    known_intent_labels.add(record["intent"])
    original_text, slot_spans = parse_annotated_utterance(record["annot_utt"])

    for slot_type, slot_start, slot_end in slot_spans:
        known_slot_types.add(slot_type)

# <PAD> = padding; <UNK> = a word absent from the training vocabulary.
word_to_id = {"<PAD>": 0, "<UNK>": 1}
for word in sorted(word_frequencies):
    word_to_id[word] = len(word_to_id)

intent_to_id = {
    intent_label: intent_index
    for intent_index, intent_label in enumerate(sorted(known_intent_labels))
}

# Construct the full BIOES tag set for every slot type.
bioes_label_names = {"O"}
for slot_type in known_slot_types:
    for tag_prefix in ("B", "I", "E", "S"):
        bioes_label_names.add(f"{tag_prefix}-{slot_type}")

slot_to_id = {
    slot_label: slot_index
    for slot_index, slot_label in enumerate(sorted(bioes_label_names))
}

id_to_intent = {index: name for name, index in intent_to_id.items()}
id_to_slot = {index: name for name, index in slot_to_id.items()}

print("Word vocabulary size:", len(word_to_id))
print("Intent classes:", len(intent_to_id))
print("Slot types:", len(known_slot_types))
print("BIOES tag classes:", len(slot_to_id))

assert len(intent_to_id) == 60, "Expected 60 MASSIVE English intents"

# Check that every dev/test label is represented in the training label space.
for partition_name in ("dev", "test"):
    for record in dataset_partitions[partition_name]:
        intent_label = record["intent"]

        if intent_label not in intent_to_id:
            raise ValueError(f"Unknown intent in {partition_name}: {intent_label}")

        plain_text, gold_slot_spans = parse_annotated_utterance(record["annot_utt"])
        gold_bioes_labels = convert_spans_to_bioes(
            tokenize_with_offsets(plain_text), gold_slot_spans
        )

        unknown_labels = [
            label for label in gold_bioes_labels if label not in slot_to_id
        ]
        if unknown_labels:
            raise ValueError(
                f"Unknown BIOES tags in {partition_name} example "
                f"{record['id']}: {unknown_labels}"
            )

print("All dev/test labels are represented by the training label mappings.")

Word vocabulary size: 5304
Intent classes: 60
Slot types: 55
BIOES tag classes: 221
All dev/test labels are represented by the training label mappings.


In [7]:
def make_slot_label_mappings(label_scheme="BIOES"):
    """Create deterministic BIOES tag-to-ID and ID-to-tag mappings."""
    if label_scheme != "BIOES":
        raise ValueError("Only BIOES slot labels are supported in this notebook")

    all_slot_labels = {"O"}
    for slot_type in known_slot_types:
        for prefix in ("B", "I", "E", "S"):
            all_slot_labels.add(f"{prefix}-{slot_type}")

    slot_label_to_id = {
        tag_name: tag_id
        for tag_id, tag_name in enumerate(sorted(all_slot_labels))
    }
    id_to_slot_label = {
        tag_id: tag_name for tag_name, tag_id in slot_label_to_id.items()
    }
    return slot_label_to_id, id_to_slot_label


def convert_spans_to_tag_scheme(token_offsets, slot_spans, label_scheme="BIOES"):
    """Compatibility helper: use direct BIOES labeling throughout."""
    if label_scheme != "BIOES":
        raise ValueError("Only BIOES slot labels are supported in this notebook")
    return convert_spans_to_bioes(token_offsets, slot_spans)


# Unit checks: one-, two-, and three-token slots.
assert convert_spans_to_bioes(
    [("London", 0, 6)], [("place_name", 0, 6)]
) == ["S-place_name"]

assert convert_spans_to_bioes(
    [("New", 0, 3), ("York", 4, 8)], [("place_name", 0, 8)]
) == ["B-place_name", "E-place_name"]

assert convert_spans_to_bioes(
    [("San", 0, 3), ("Francisco", 4, 13), ("Bay", 14, 17)],
    [("place_name", 0, 17)],
) == ["B-place_name", "I-place_name", "E-place_name"]

print("BIOES annotation conversion checks passed.")

BIOES annotation conversion checks passed.


In [8]:
def decode_tagged_spans(predicted_tags, token_offsets, utterance, label_scheme="BIOES"):
    """Turn BIOES tag strings back into (slot_type, original_text) pairs."""
    if label_scheme != "BIOES":
        raise ValueError("Only BIOES decoding is supported")

    extracted_slots = []
    active_slot_type = None
    active_start = None
    active_end = None

    def close_active_slot():
        nonlocal active_slot_type, active_start, active_end

        if active_slot_type is not None:
            original_slot_text = utterance[active_start:active_end]
            extracted_slots.append((active_slot_type, original_slot_text))

        active_slot_type = None
        active_start = None
        active_end = None

    for tag, (token_start, token_end) in zip(predicted_tags, token_offsets):
        if tag == "O":
            close_active_slot()
            continue

        tag_prefix, slot_type = tag.split("-", 1)

        if tag_prefix == "S":
            close_active_slot()
            extracted_slots.append((slot_type, utterance[token_start:token_end]))

        elif tag_prefix == "B":
            close_active_slot()
            active_slot_type = slot_type
            active_start = token_start
            active_end = token_end

        elif tag_prefix == "I":
            if active_slot_type != slot_type:
                close_active_slot()
                active_slot_type = slot_type
                active_start = token_start
            active_end = token_end

        elif tag_prefix == "E":
            if active_slot_type != slot_type:
                close_active_slot()
                active_slot_type = slot_type
                active_start = token_start
            active_end = token_end
            close_active_slot()

        else:
            raise ValueError(f"Unknown BIOES prefix: {tag_prefix}")

    close_active_slot()
    return extracted_slots


def is_valid_transition(previous_tag, next_tag, scheme):
    """Decide whether moving from one BIOES tag to another is legal."""
    if scheme != "BIOES":
        raise ValueError("The constrained CRF supports BIOES only")

    previous_is_complete = (
        previous_tag == "O" or previous_tag.startswith(("E-", "S-"))
    )

    if previous_is_complete:
        return next_tag == "O" or next_tag.startswith(("B-", "S-"))

    previous_prefix, previous_slot_type = previous_tag.split("-", 1)
    return next_tag in (
        f"I-{previous_slot_type}",
        f"E-{previous_slot_type}",
    )


class ConstrainedBIOESCRF(nn.Module):
    """Linear-chain CRF that disallows impossible BIOES label sequences.

    `transition_scores[previous_id, next_id]` is a learned preference for
    adjacent tags. Large negative scores are used for illegal transitions.
    """

    def __init__(self, tag_names):
        super().__init__()

        self.tag_names = tag_names
        self.tag_count = len(tag_names)

        # Keep these names unchanged for saved-checkpoint compatibility.
        self.transition_scores = nn.Parameter(
            torch.zeros(self.tag_count, self.tag_count)
        )
        self.start_scores = nn.Parameter(torch.zeros(self.tag_count))
        self.end_scores = nn.Parameter(torch.zeros(self.tag_count))

        legal_transitions = torch.tensor(
            [
                [
                    is_valid_transition(previous_tag, next_tag, "BIOES")
                    for next_tag in tag_names
                ]
                for previous_tag in tag_names
            ],
            dtype=torch.bool,
        )

        legal_start_tags = torch.tensor(
            [
                tag == "O" or tag.startswith(("B-", "S-"))
                for tag in tag_names
            ],
            dtype=torch.bool,
        )

        legal_end_tags = torch.tensor(
            [
                tag == "O" or tag.startswith(("E-", "S-"))
                for tag in tag_names
            ],
            dtype=torch.bool,
        )

        self.register_buffer("legal_transition", legal_transitions)
        self.register_buffer("legal_start", legal_start_tags)
        self.register_buffer("legal_end", legal_end_tags)

    def constrained_scores(self):
        """Return transition, start, and end scores after applying BIOES rules."""
        illegal_score = -10_000.0

        transition_scores = self.transition_scores.masked_fill(
            ~self.legal_transition, illegal_score
        )
        start_scores = self.start_scores.masked_fill(
            ~self.legal_start, illegal_score
        )
        end_scores = self.end_scores.masked_fill(
            ~self.legal_end, illegal_score
        )
        return transition_scores, start_scores, end_scores

    def negative_log_likelihood(self, emissions, gold_tags, lengths):
        """Compute the CRF training loss, averaged across batch examples.

        Loss = log(sum(scores of all legal paths)) - score(gold path).
        """
        transition_scores, start_scores, end_scores = self.constrained_scores()
        example_losses = []

        for example_index, sequence_length in enumerate(lengths.tolist()):
            token_emissions = emissions[example_index, :sequence_length]
            gold_sequence = gold_tags[example_index, :sequence_length]

            # Forward algorithm: compute log-sum-exp over all paths.
            forward_scores = start_scores + token_emissions[0]

            for token_index in range(1, sequence_length):
                candidate_scores = (
                    forward_scores.unsqueeze(1) + transition_scores
                )
                forward_scores = (
                    torch.logsumexp(candidate_scores, dim=0)
                    + token_emissions[token_index]
                )

            log_partition = torch.logsumexp(
                forward_scores + end_scores, dim=0
            )

            # Score the actual gold BIOES path.
            first_gold_tag = gold_sequence[0]
            gold_path_score = (
                start_scores[first_gold_tag]
                + token_emissions[0, first_gold_tag]
            )

            for token_index in range(1, sequence_length):
                previous_tag = gold_sequence[token_index - 1]
                current_tag = gold_sequence[token_index]

                gold_path_score = gold_path_score + transition_scores[
                    previous_tag, current_tag
                ]
                gold_path_score = gold_path_score + token_emissions[
                    token_index, current_tag
                ]

            gold_path_score = gold_path_score + end_scores[gold_sequence[-1]]
            example_losses.append(log_partition - gold_path_score)

        return torch.stack(example_losses).mean()

    @torch.no_grad()
    def decode(self, emissions, lengths):
        """Decode the best legal BIOES tag sequence using Viterbi."""
        transition_scores, start_scores, end_scores = self.constrained_scores()
        decoded_sequences = []

        for example_index, sequence_length in enumerate(lengths.tolist()):
            token_emissions = emissions[example_index, :sequence_length]
            best_scores = start_scores + token_emissions[0]
            back_pointers = []

            # Store the best prior tag for each possible current tag.
            for token_index in range(1, sequence_length):
                candidate_scores = (
                    best_scores.unsqueeze(1) + transition_scores
                )
                best_previous_scores, best_previous_tags = candidate_scores.max(dim=0)
                best_scores = best_previous_scores + token_emissions[token_index]
                back_pointers.append(best_previous_tags)

            best_final_tag = int((best_scores + end_scores).argmax().item())
            best_path = [best_final_tag]

            for previous_tag_choices in reversed(back_pointers):
                preceding_tag = int(previous_tag_choices[best_path[-1]].item())
                best_path.append(preceding_tag)

            decoded_sequences.append(list(reversed(best_path)))

        return decoded_sequences


print("BIOES-constrained CRF implementation ready.")

BIOES-constrained CRF implementation ready.


## 4. BPE tokenization, character vocabulary, and padded data batches

M1 uses word IDs. M2 adds **raw characters per token** (up to 32 characters). M3 combines its separate GloVe lookup with BPE subwords. All three models share original word boundaries and BIOES labels, so their slot-emission tensors can be fused.

The original checkpoint ID mappings are checked before evaluation. Changing vocabulary order would make a checkpoint incompatible even when the model shapes are unchanged.


In [9]:
# Every joint model must predict slot tags in the same BIOES ID order.
bioes_label_to_id, bioes_id_to_label = make_slot_label_mappings("BIOES")
TAG_NAMES = [bioes_id_to_label[tag_id] for tag_id in range(len(bioes_id_to_label))]

# Load only the checkpoints requested in Section 1.
loaded = {}
for model_name, selected_mode in MODE.items():
    if selected_mode != "load":
        continue

    checkpoint = torch.load(
        LOAD_PATH[model_name], map_location="cpu", weights_only=True
    )
    loaded[model_name] = checkpoint

    # Matching the ID mappings is essential for valid predictions.
    if checkpoint["intent_to_id"] != intent_to_id:
        raise ValueError(f"{model_name}: intent-label mapping mismatch")

    if model_name == "M3":
        if checkpoint["bioes_label_to_id"] != bioes_label_to_id:
            raise ValueError("M3 BIOES-label mapping mismatch")
    else:
        if checkpoint["word_to_id"] != word_to_id:
            raise ValueError(f"{model_name}: word vocabulary mismatch")
        if checkpoint["label_to_id"] != bioes_label_to_id:
            raise ValueError(f"{model_name}: BIOES-label mapping mismatch")

    if model_name == "M2" and checkpoint.get("architecture") != "CharacterCNN":
        raise ValueError(
            "M2 needs the verified CharacterCNN checkpoint, "
            "not the older Transformer checkpoint."
        )

    print(f"Loaded metadata for {model_name}: {LOAD_PATH[model_name]}")

# M3's BPE tokenizer must be exactly the same as during its training.
if MODE["M3"] == "load":
    bpe = Tokenizer.from_file(str(BPE_LOAD_PATH))
    print("Loaded existing BPE tokenizer:", BPE_LOAD_PATH)
else:
    bpe = Tokenizer(models.BPE(unk_token="<unk>"))
    bpe.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    bpe.decoder = decoders.ByteLevel()

    bpe_trainer = trainers.BpeTrainer(
        vocab_size=BPE_VOCAB_SIZE,
        min_frequency=2,
        special_tokens=["<pad>", "<unk>"],
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    )
    bpe.train_from_iterator(
        (record["utt"] for record in training_records),
        trainer=bpe_trainer,
    )
    print("Trained BPE tokenizer using training data only")

BPE_PAD = bpe.token_to_id("<pad>")
BPE_UNK = bpe.token_to_id("<unk>")
if BPE_PAD is None or BPE_UNK is None:
    raise ValueError("BPE tokenizer needs <pad> and <unk> tokens")

BPE_SAVE_PATH = OUTPUT_DIR / "M3_bpe_tokenizer.json"
bpe.save(str(BPE_SAVE_PATH))


def word_pieces(word):
    """Convert one word to BPE token IDs (used for M3)."""
    piece_ids = bpe.encode(word, add_special_tokens=False).ids
    if not piece_ids:
        raise ValueError(f"No BPE pieces for {word!r}")
    return piece_ids


def build_train_character_vocab():
    """Match the original M2 character vocabulary's first-seen ordering."""
    character_vocabulary = {"<PAD>": 0, "<UNK>": 1}
    for record in training_records:
        for character in record["utt"].lower():
            if character not in character_vocabulary:
                character_vocabulary[character] = len(character_vocabulary)
    return character_vocabulary


if "M2" in loaded:
    # Saved checkpoints have their own required character ID mapping.
    char_to_id = loaded["M2"]["extra_vocab"]
    expected_character_rows = loaded["M2"]["weights"]["character_embedding.weight"].shape[0]
    if len(char_to_id) != expected_character_rows:
        raise ValueError("M2 checkpoint character vocabulary size mismatch")
else:
    char_to_id = build_train_character_vocab()

CHARACTER_LIMIT = 32
print("BIOES tag count:", len(TAG_NAMES))
print("BPE vocabulary size:", bpe.get_vocab_size())
print("M2 character vocabulary size:", len(char_to_id))


Trained BPE tokenizer using training data only
BIOES tag count: 221
BPE vocabulary size: 2000
M2 character vocabulary size: 35


In [10]:
class JointDataset(Dataset):
    """Prepare every utterance for all three joint intent-slot models."""

    def __init__(self, records):
        self.examples = []

        for record in records:
            utterance = record["utt"]
            recovered_text, gold_slot_spans = parse_annotated_utterance(
                record["annot_utt"]
            )

            if recovered_text != utterance:
                raise ValueError("Annotation text does not match the original utterance")

            token_offsets = tokenize_with_offsets(utterance)
            words = [token for token, _, _ in token_offsets]

            if not words:
                raise ValueError("Found an empty utterance")

            bioes_tags = convert_spans_to_bioes(token_offsets, gold_slot_spans)
            gold_slot_ids = [bioes_label_to_id[tag] for tag in bioes_tags]
            gold_slot_pairs = [
                (slot_type, utterance[start:end])
                for slot_type, start, end in gold_slot_spans
            ]

            self.examples.append({
                "words": words,
                "pieces": [word_pieces(word) for word in words],
                "slot_ids": gold_slot_ids,
                "intent_id": intent_to_id[record["intent"]],
                "utterance": utterance,
                "token_offsets": [
                    (start, end) for _, start, end in token_offsets
                ],
                "gold_slot_pairs": gold_slot_pairs,
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


train_dataset = JointDataset(training_records)
dev_dataset = JointDataset(validation_records)

# IMPORTANT: M3 frozen GloVe vocabulary is built from TRAIN only.
# Never use dev or test utterances to decide which words get embedding rows.
all_words = sorted({
    word
    for example in train_dataset.examples
    for word in example["words"]
})

word_index = {
    word: index + 1 for index, word in enumerate(all_words)
}
print(f"M3 train-only vocabulary: {len(word_index):,} words")

if "M3" in loaded:
    checkpoint = loaded["M3"]

    if checkpoint["word_index"] != word_index:
        raise ValueError(
            "M3 checkpoint vocabulary mismatch. Older E15 weights used train+dev. "
            "Retrain M3 using the train-only vocabulary and load its new checkpoint."
        )
    if checkpoint["config"]["bpe_vocab_size"] != bpe.get_vocab_size():
        raise ValueError("M3 BPE vocabulary size mismatch")


def collate(examples):
    """Pad utterances in one batch so models can process them together.

    Shapes:
    - word_ids, glove_ids, piece_counts, slot_ids: [batch, words]
    - pieces: [batch, words, subwords]
    - char_ids: [batch, words, characters]
    """
    batch_size = len(examples)
    maximum_words = max(len(example["words"]) for example in examples)
    maximum_subwords = max(
        len(subwords)
        for example in examples
        for subwords in example["pieces"]
    )

    lengths = torch.tensor(
        [len(example["words"]) for example in examples],
        dtype=torch.long,
    )

    pieces = torch.full(
        (batch_size, maximum_words, maximum_subwords),
        fill_value=BPE_PAD,
        dtype=torch.long,
    )
    piece_counts = torch.zeros((batch_size, maximum_words), dtype=torch.long)

    word_ids = torch.full(
        (batch_size, maximum_words),
        fill_value=word_to_id["<PAD>"],
        dtype=torch.long,
    )
    glove_ids = torch.zeros((batch_size, maximum_words), dtype=torch.long)
    char_ids = torch.zeros(
        (batch_size, maximum_words, CHARACTER_LIMIT),
        dtype=torch.long,
    )

    # The -100 entries indicate padding for the gold slot label sequence.
    slot_ids = torch.full((batch_size, maximum_words), -100, dtype=torch.long)

    for example_index, example in enumerate(examples):
        word_count = len(example["words"])
        slot_ids[example_index, :word_count] = torch.tensor(example["slot_ids"])

        for word_position, word in enumerate(example["words"]):
            # M3: subword IDs for this original word.
            subword_ids = example["pieces"][word_position]
            number_of_subwords = len(subword_ids)

            pieces[
                example_index, word_position, :number_of_subwords
            ] = torch.tensor(subword_ids)
            piece_counts[example_index, word_position] = number_of_subwords

            # M1/M2: training-derived word IDs.
            word_ids[example_index, word_position] = word_to_id.get(
                word.lower(), word_to_id["<UNK>"]
            )

            # M3: lookup in its original word-vector table.
            # An unseen word safely maps to index 0 (the zero vector).
            glove_ids[example_index, word_position] = word_index.get(word, 0)

            # M2: raw characters from the current utterance.
            for character_index, character in enumerate(
                word.lower()[:CHARACTER_LIMIT]
            ):
                char_ids[example_index, word_position, character_index] = (
                    char_to_id.get(character, 1)
                )

    return {
        "lengths": lengths,
        "word_ids": word_ids,
        "glove_ids": glove_ids,
        "char_ids": char_ids,
        "pieces": pieces,
        "piece_counts": piece_counts,
        "slot_ids": slot_ids,
        "intent_ids": torch.tensor(
            [example["intent_id"] for example in examples], dtype=torch.long
        ),
        "words": [example["words"] for example in examples],
        "utterances": [example["utterance"] for example in examples],
        "token_offsets": [example["token_offsets"] for example in examples],
        "gold_slot_pairs": [example["gold_slot_pairs"] for example in examples],
    }


def make_loader(dataset, batch_size, shuffle=False):
    """Create a PyTorch DataLoader using the shared padding function."""
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=collate,
        num_workers=0,
    )


train_loader = make_loader(train_dataset, BATCH_SIZE, shuffle=True)
dev_loader = make_loader(dev_dataset, BATCH_SIZE, shuffle=False)

# Slightly larger batches for the word-only models.
m1_train_loader = make_loader(train_dataset, M1_BATCH_SIZE, shuffle=True)
m1_dev_loader = make_loader(dev_dataset, M1_BATCH_SIZE, shuffle=False)

print("Training examples:", len(train_dataset))
print("Validation examples:", len(dev_dataset))

M3 train-only vocabulary: 5,302 words
Training examples: 11514
Validation examples: 2033


## 5. Initialize permitted GloVe vectors only when training requires them

A loaded checkpoint contains its embedding weights, so the GloVe text file is optional when all models are loaded. M1's initial phase uses **trainable** GloVe-initialized word embeddings. M3 uses a **frozen** GloVe table plus trainable BPE embeddings. These choices match the original experiments.


In [11]:
# ---------------------------------------------------------------------------
# Decide whether any selected training path needs fresh GloVe word vectors.
# ---------------------------------------------------------------------------
need_m1_glove = (MODE["M1"] == "train" and M1_INIT_MODE == "train")
need_m3_glove = MODE["M3"] == "train"

m1_initial_matrix = None
m3_glove_matrix = None

if "M3" in loaded:
    # No GloVe file is needed for a loaded M3: its checkpoint stores the matrix.
    m3_glove_matrix = loaded["M3"]["weights"]["word_embedding.weight"].clone().float()

if need_m1_glove or need_m3_glove:
    if not GLOVE_PATH.is_file():
        raise FileNotFoundError(f"Missing GloVe word embeddings: {GLOVE_PATH}")

    # Build a list of useful words before scanning the large GloVe text file.
    wanted_words = set()

    if need_m1_glove:
        wanted_words.update(word_to_id.keys())

    if need_m3_glove:
        wanted_words.update(word.lower() for word in all_words)

    glove_lookup = {}
    timer = start_clock("GloVe loading")

    with GLOVE_PATH.open("r", encoding="utf-8") as glove_file:
        for line in glove_file:
            fields = line.rstrip().split(" ")

            # 300D GloVe: one word followed by exactly 300 float values.
            if len(fields) != 301:
                continue

            word = fields[0]
            if word in wanted_words:
                glove_lookup[word] = np.asarray(fields[1:], dtype=np.float32)

    stop_clock("GloVe loading", timer, action="preprocessing")
    print(f"GloVe coverage: {len(glove_lookup)} / {len(wanted_words)} words")

    if need_m1_glove:
        # M1 uses this trainable initial word embedding matrix.
        seed_everything(SEED)
        m1_initial_matrix = torch.empty((len(word_to_id), 300))
        nn.init.normal_(m1_initial_matrix, mean=0.0, std=0.05)
        m1_initial_matrix[0].zero_()  # Padding embedding.

        for word, word_id in word_to_id.items():
            pretrained_vector = glove_lookup.get(word)
            if pretrained_vector is not None:
                m1_initial_matrix[word_id] = torch.from_numpy(pretrained_vector)

    if need_m3_glove:
        # M3 uses a separate word_index table and a zero vector for index 0.
        m3_word_vectors = np.zeros(
            (len(word_index) + 1, 300), dtype=np.float32
        )

        for word, word_id in word_index.items():
            pretrained_vector = glove_lookup.get(word.lower())
            if pretrained_vector is not None:
                m3_word_vectors[word_id] = pretrained_vector

        m3_glove_matrix = torch.from_numpy(m3_word_vectors)

    del glove_lookup

if m3_glove_matrix is None:
    raise RuntimeError("Failed to initialize or load M3's GloVe embedding matrix")

expected_matrix_shape = (len(word_index) + 1, 300)
if m3_glove_matrix.shape != expected_matrix_shape:
    raise ValueError(
        f"M3 GloVe matrix shape mismatch: {m3_glove_matrix.shape} "
        f"versus {expected_matrix_shape}"
    )

print("Word embedding matrices are ready.")

[2026-11-11T17:16:43+08:00] START GloVe loading
[2026-11-11T17:16:55+08:00] END GloVe loading | elapsed 0:00:12
GloVe coverage: 4918 / 5304 words
Word embedding matrices are ready.


## 6. Define three **joint** models and a BIOES-constrained CRF

All models produce two outputs: `intent_logits` (one prediction per utterance) and `slot_emissions` (one set of BIOES scores per word). The CRF learns how tags relate across the entire utterance.

Layer attribute names (`encoder`, `intent_head`, `slot_head`, `crf`, etc.) are deliberately preserved so **historical `.pt` state dictionaries still load with `strict=True`**.


In [12]:
class M1Joint(nn.Module):
    """M1: one-layer BiLSTM with shared features for intent and slot heads."""

    def __init__(self):
        super().__init__()

        # Names of these layers are preserved for saved-checkpoint loading.
        self.word_embedding = nn.Embedding(
            num_embeddings=len(word_to_id),
            embedding_dim=300,
            padding_idx=0,
        )
        self.encoder = nn.LSTM(
            input_size=300,
            hidden_size=128,
            batch_first=True,
            bidirectional=True,
        )

        # BiLSTM output is 128 forward + 128 backward = 256 dimensions.
        self.dropout = nn.Dropout(0.3)
        self.attention_layer = nn.Linear(256, 1)
        self.intent_head = nn.Linear(256, len(intent_to_id))
        self.slot_head = nn.Linear(256, len(TAG_NAMES))
        self.crf = ConstrainedBIOESCRF(TAG_NAMES)

    def encode(self, word_ids, lengths, embeddings=None):
        """Apply the shared encoder and produce intent and slot logits.

        `embeddings` allows M2 to supply their own character-enhanced
        word representations without changing the shared BiLSTM logic.
        """
        if embeddings is None:
            token_embeddings = self.word_embedding(word_ids)
        else:
            token_embeddings = embeddings

        token_embeddings = self.dropout(token_embeddings)

        # Packing prevents the BiLSTM from interpreting padding as real words.
        packed_embeddings = pack_padded_sequence(
            token_embeddings,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        packed_outputs, _ = self.encoder(packed_embeddings)
        token_features, _ = pad_packed_sequence(
            packed_outputs,
            batch_first=True,
            total_length=word_ids.shape[1],
        )

        # Attention pooling forms one sentence representation for intent.
        word_positions = torch.arange(
            token_features.shape[1], device=token_features.device
        )
        valid_word_mask = (
            word_positions.unsqueeze(0)
            < lengths.to(token_features.device).unsqueeze(1)
        )

        attention_scores = self.attention_layer(token_features).squeeze(-1)
        attention_scores = attention_scores.masked_fill(
            ~valid_word_mask, -10_000.0
        )
        attention_weights = attention_scores.softmax(dim=1)
        sentence_features = (
            token_features * attention_weights.unsqueeze(-1)
        ).sum(dim=1)

        # Intent uses the sentence vector; slot tags use each word's vector.
        intent_logits = self.intent_head(self.dropout(sentence_features))
        slot_emissions = self.slot_head(self.dropout(token_features))

        return intent_logits, slot_emissions

    def forward(self, word_ids, lengths):
        return self.encode(word_ids, lengths)


class M2CharacterJoint(M1Joint):
    """M2: Character CNN with projected residual addition to word embeddings.

    The name and structure match the verified CharacterCNN E12 checkpoint.
    """

    def __init__(self):
        super().__init__()

        self.character_embedding = nn.Embedding(
            len(char_to_id), 32, padding_idx=0
        )
        self.character_convolution = nn.Conv1d(
            in_channels=32, out_channels=96, kernel_size=3, padding=1
        )
        self.character_projection = nn.Linear(96, 300)

        # Start with no change to the original M1 word representation.
        nn.init.zeros_(self.character_projection.weight)
        nn.init.zeros_(self.character_projection.bias)

    def forward(self, word_ids, lengths, char_ids):
        batch_size, word_count, characters_per_word = char_ids.shape

        # [batch, words, characters, 32]
        character_vectors = self.character_embedding(char_ids)

        # Conv1d expects [batch * words, embedding_channels, characters].
        character_vectors = character_vectors.reshape(
            batch_size * word_count, characters_per_word, 32
        ).transpose(1, 2)

        convolution_output = self.character_convolution(character_vectors)
        pooled_character_features = convolution_output.relu().max(dim=-1).values

        character_features = self.character_projection(
            pooled_character_features.reshape(batch_size, word_count, 96)
        )

        # RESIDUAL ADDITION: 300D GloVe + 300D projected characters.
        word_embeddings = self.word_embedding(word_ids)
        enhanced_embeddings = word_embeddings + character_features

        return self.encode(word_ids, lengths, enhanced_embeddings)


class M3Joint(nn.Module):
    """M3: GloVe plus learned BPE embeddings, followed by a joint BiLSTM."""

    def __init__(self, glove_matrix):
        super().__init__()

        self.bpe_embedding = nn.Embedding(
            bpe.get_vocab_size(), SUBWORD_DIM, padding_idx=BPE_PAD
        )

        # Original M3 uses frozen pretrained GloVe vectors.
        self.word_embedding = nn.Embedding.from_pretrained(
            glove_matrix.clone(),
            freeze=True,
            padding_idx=0,
        )

        self.encoder = nn.LSTM(
            input_size=300 + SUBWORD_DIM,
            hidden_size=M3_HIDDEN_SIZE,
            num_layers=M3_LAYERS,
            batch_first=True,
            bidirectional=True,
            dropout=M3_DROPOUT if M3_LAYERS > 1 else 0,
        )

        encoder_output_size = 2 * M3_HIDDEN_SIZE
        self.dropout = nn.Dropout(M3_DROPOUT)
        self.attention = nn.Linear(encoder_output_size, 1)
        self.intent_head = nn.Linear(encoder_output_size, len(intent_to_id))
        self.slot_head = nn.Linear(encoder_output_size, len(TAG_NAMES))
        self.crf = ConstrainedBIOESCRF(TAG_NAMES)

    def forward(self, batch):
        subword_ids = batch["pieces"].to(device)
        subword_counts = batch["piece_counts"].to(device)
        word_lengths = batch["lengths"]

        # Keep only valid subword positions when computing each word vector.
        subword_positions = torch.arange(
            subword_ids.shape[-1], device=device
        )
        valid_subword_mask = (
            subword_positions.view(1, 1, -1)
            < subword_counts.unsqueeze(-1)
        )

        embedded_subwords = self.bpe_embedding(subword_ids)
        summed_subwords = (
            embedded_subwords * valid_subword_mask.unsqueeze(-1)
        ).sum(dim=2)
        averaged_subwords = (
            summed_subwords / subword_counts.clamp_min(1).unsqueeze(-1)
        )

        glove_embeddings = self.word_embedding(batch["glove_ids"].to(device))

        # CONCATENATION: 128D BPE + 300D GloVe = 428D input.
        token_embeddings = torch.cat(
            [averaged_subwords, glove_embeddings], dim=-1
        )
        token_embeddings = self.dropout(token_embeddings)

        packed_embeddings = pack_padded_sequence(
            token_embeddings,
            word_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        packed_outputs, _ = self.encoder(packed_embeddings)
        token_features, _ = pad_packed_sequence(
            packed_outputs,
            batch_first=True,
            total_length=token_embeddings.shape[1],
        )

        # Attention pooling for utterance-level intent classification.
        word_positions = torch.arange(
            token_embeddings.shape[1], device=device
        )
        valid_word_mask = (
            word_positions.unsqueeze(0) < word_lengths.to(device).unsqueeze(1)
        )

        attention_scores = self.attention(token_features).squeeze(-1)
        attention_scores = attention_scores.masked_fill(
            ~valid_word_mask, -10_000.0
        )
        attention_weights = attention_scores.softmax(dim=1)
        sentence_features = (
            token_features * attention_weights.unsqueeze(-1)
        ).sum(dim=1)

        intent_logits = self.intent_head(self.dropout(sentence_features))
        slot_emissions = self.slot_head(self.dropout(token_features))

        return intent_logits, slot_emissions




def model_outputs(model, batch, name):
    """Call the correct joint model while keeping a common output format."""
    if name == "M1":
        return model(batch["word_ids"].to(device), batch["lengths"])

    if name == "M2":
        return model(
            batch["word_ids"].to(device),
            batch["lengths"],
            batch["char_ids"].to(device),
        )

    if name == "M3":
        return model(batch)

    raise KeyError(f"Unknown joint model: {name}")


print("Joint architectures ready: M1, M2, M3")


Joint architectures ready: M1, M2, M3


## 7. Training, validation, checkpoints, and elapsed-time logs

The **joint objective** is `intent cross-entropy + CRF negative log-likelihood`. Every epoch evaluates the **dev** split; the highest combined validation score is saved, and patience stops training after repeated non-improvement.

Training restores the best checkpoint before returning. Loading a checkpoint skips gradients entirely. Each epoch prints start/end timestamps, duration, intent accuracy, slot F1, and combined score.


In [13]:
def span_counts(predicted, actual):
    """Count exact-span true positives, false positives and false negatives.

    Both inputs are lists of (slot_type, original_slot_text) pairs. Counter
    preserves duplicate occurrences, matching the previous evaluation logic.
    """
    predicted_counter = Counter(predicted)
    gold_counter = Counter(actual)

    correct_slots = sum((predicted_counter & gold_counter).values())
    incorrect_slots = sum(predicted_counter.values()) - correct_slots
    missed_slots = sum(gold_counter.values()) - correct_slots

    return correct_slots, incorrect_slots, missed_slots


def exact_micro_f1(true_positive, false_positive, false_negative):
    """Calculate exact-span slot micro-F1 on a 0–100 scale."""
    denominator = (
        2 * true_positive + false_positive + false_negative
    )

    if denominator == 0:
        return 100.0

    return 100.0 * (2 * true_positive) / denominator


@torch.inference_mode()
def evaluate_model(model, loader, name):
    """Measure a single joint model's intent accuracy and slot micro-F1."""
    model.eval()

    correct_intents = 0
    evaluated_examples = 0
    true_positive = 0
    false_positive = 0
    false_negative = 0

    for batch in loader:
        intent_logits, slot_emissions = model_outputs(model, batch, name)
        predicted_slot_paths = model.crf.decode(
            slot_emissions, batch["lengths"]
        )

        predicted_intent_ids = intent_logits.argmax(dim=-1).cpu()
        correct_intents += int(
            (predicted_intent_ids == batch["intent_ids"]).sum().item()
        )
        evaluated_examples += len(batch["intent_ids"])

        for example_index, predicted_tag_ids in enumerate(predicted_slot_paths):
            predicted_tag_names = [
                TAG_NAMES[tag_id] for tag_id in predicted_tag_ids
            ]

            predicted_spans = decode_tagged_spans(
                predicted_tag_names,
                batch["token_offsets"][example_index],
                batch["utterances"][example_index],
                label_scheme="BIOES",
            )
            actual_spans = batch["gold_slot_pairs"][example_index]

            matches, extras, misses = span_counts(predicted_spans, actual_spans)
            true_positive += matches
            false_positive += extras
            false_negative += misses

    intent_accuracy = 100.0 * correct_intents / evaluated_examples
    slot_f1 = exact_micro_f1(
        true_positive, false_positive, false_negative
    )

    return {
        "intent_accuracy": intent_accuracy,
        "slot_f1": slot_f1,
        "combined_score": (intent_accuracy + slot_f1) / 2.0,
    }


@torch.inference_mode()
def evaluate_validation_loss(model, loader, name):
    """Compute development loss without changing the model weights.

    The loss matches training: intent cross-entropy + BIOES CRF negative
    log-likelihood. Both component losses are batch-size-weighted means.
    """
    model.eval()

    summed_intent_loss = 0.0
    summed_slot_loss = 0.0
    total_examples = 0

    for batch in loader:
        intent_logits, slot_emissions = model_outputs(model, batch, name)

        intent_loss = F.cross_entropy(
            intent_logits, batch["intent_ids"].to(device)
        )
        slot_loss = model.crf.negative_log_likelihood(
            slot_emissions,
            batch["slot_ids"].to(device),
            batch["lengths"],
        )

        batch_size = len(batch["intent_ids"])
        summed_intent_loss += intent_loss.item() * batch_size
        summed_slot_loss += slot_loss.item() * batch_size
        total_examples += batch_size

    mean_intent_loss = summed_intent_loss / total_examples
    mean_slot_loss = summed_slot_loss / total_examples

    return {
        "val_intent_loss": mean_intent_loss,
        "val_slot_loss": mean_slot_loss,
        "val_loss": mean_intent_loss + mean_slot_loss,
    }


def make_checkpoint(name, model, epoch, score, history):
    """Create one self-describing checkpoint for a trained joint model."""
    # Clone to CPU so that torch.save does not depend on GPU memory.
    current_weights = {
        tensor_name: tensor.detach().cpu().clone()
        for tensor_name, tensor in model.state_dict().items()
    }

    checkpoint = {
        "weights": current_weights,
        "best_epoch": epoch,
        "seed": SEED,
        "parameter_count": sum(
            parameter.numel() for parameter in model.parameters()
        ),
        "best_validation_score": score,
        "history": list(history),
        "intent_to_id": intent_to_id,
        "label_to_id": bioes_label_to_id,
        "word_to_id": word_to_id,
        "model_name": name,
    }

    if name == "M2":
        checkpoint["architecture"] = "CharacterCNN"
        checkpoint["extra_vocab"] = char_to_id

    if name == "M3":
        checkpoint["bioes_label_to_id"] = bioes_label_to_id
        checkpoint["word_index"] = word_index
        checkpoint["best_dev_score"] = score
        checkpoint["config"] = {
            "bpe_vocab_size": bpe.get_vocab_size(),
            "subword_dim": SUBWORD_DIM,
            "hidden_size": M3_HIDDEN_SIZE,
            "lstm_layers": M3_LAYERS,
            "dropout": M3_DROPOUT,
        }

    return checkpoint


def train_joint(
    name,
    model,
    train_batches,
    dev_batches,
    lr,
    max_epochs,
    patience,
    checkpoint_file,
):
    """Train a joint model and restore its best validation checkpoint.

    A training epoch:
      1. Predict intent logits and slot emissions.
      2. Add intent cross-entropy and slot CRF negative log-likelihood.
      3. Backpropagate and update parameters.
      4. Evaluate on dev, save if improved, or advance early stopping.
    """
    training_clock = start_clock(f"{name} training")
    # M1_init is the preliminary phase of M1, not a separate ensemble model.
    prediction_model_name = "M1" if name == "M1_init" else name
    model = model.to(device)

    trainable_parameters = [
        parameter for parameter in model.parameters()
        if parameter.requires_grad
    ]
    optimizer = torch.optim.AdamW(
        trainable_parameters,
        lr=lr,
        weight_decay=WEIGHT_DECAY,
    )

    best_validation_score = -float("inf")
    epochs_without_improvement = 0
    training_history = []

    for epoch in range(1, max_epochs + 1):
        epoch_clock = start_clock(f"{name} epoch {epoch:02d}")
        model.train()
        weighted_loss_sum = 0.0
        number_of_training_examples = 0

        for batch in train_batches:
            optimizer.zero_grad(set_to_none=True)
            intent_logits, slot_emissions = model_outputs(model, batch, prediction_model_name)

            intent_loss = F.cross_entropy(
                intent_logits,
                batch["intent_ids"].to(device),
            )
            slot_loss = model.crf.negative_log_likelihood(
                slot_emissions,
                batch["slot_ids"].to(device),
                batch["lengths"],
            )
            total_loss = intent_loss + slot_loss

            total_loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            current_batch_size = len(batch["intent_ids"])
            weighted_loss_sum += total_loss.item() * current_batch_size
            number_of_training_examples += current_batch_size

        # Choose checkpoints on validation rather than training accuracy.
        validation_result = evaluate_model(model, dev_batches, prediction_model_name)
        validation_losses = evaluate_validation_loss(
            model, dev_batches, prediction_model_name
        )
        elapsed_seconds = stop_clock(
            f"{name} epoch {epoch:02d}",
            epoch_clock,
            action="epoch",
        )

        combined_score = validation_result["combined_score"]
        mean_training_loss = weighted_loss_sum / number_of_training_examples

        training_history.append({
            "epoch": epoch,
            "train_loss": mean_training_loss,
            **validation_losses,
            "elapsed_seconds": round(elapsed_seconds, 2),
            **validation_result,
        })

        print(
            f"{name} epoch {epoch:02d} | "
            f"train loss {mean_training_loss:.4f} | "
            f"dev loss {validation_losses['val_loss']:.4f} | "
            f"intent {validation_result['intent_accuracy']:.2f}% | "
            f"slot F1 {validation_result['slot_f1']:.2f}% | "
            f"combined {combined_score:.2f}",
            flush=True,
        )

        if combined_score > best_validation_score + 1e-6:
            best_validation_score = combined_score
            epochs_without_improvement = 0

            checkpoint = make_checkpoint(
                prediction_model_name, model, epoch, best_validation_score, training_history
            )
            torch.save(checkpoint, checkpoint_file)
            print("  Saved new best checkpoint:", checkpoint_file)
        else:
            epochs_without_improvement += 1

            if epochs_without_improvement >= patience:
                print(f"Early stopping after {epoch} epochs.")
                break

    # Preserve ALL epoch records, not only the epochs before the best checkpoint.
    history_file = OUTPUT_DIR / f"{name}_epoch_history.csv"
    pd.DataFrame(training_history).to_csv(history_file, index=False)
    print(f"Saved epoch training and validation losses: {history_file}")

    stop_clock(f"{name} training", training_clock, action="train")

    # The final model must contain the BEST epoch's weights, not necessarily
    # the last epoch's weights.
    best_checkpoint = torch.load(
        checkpoint_file, map_location="cpu", weights_only=True
    )
    model.load_state_dict(best_checkpoint["weights"], strict=True)
    model.eval()

    return model, best_checkpoint


def load_joint(name, model, path):
    """Skip training and load a verified historical checkpoint."""
    loading_clock = start_clock(f"{name} loading")

    checkpoint = (
        loaded[name]
        if name in loaded
        else torch.load(path, map_location="cpu", weights_only=True)
    )
    model = model.to(device)
    model.load_state_dict(checkpoint["weights"], strict=True)
    model.eval()

    stop_clock(f"{name} loading", loading_clock, action="load")
    print(f"Loaded {name} checkpoint: {path}")
    return model, checkpoint


def checkpoint_best_score(name, checkpoint):
    """Read the recorded dev score using the correct checkpoint field name."""
    if name == "M3":
        return checkpoint.get(
            "best_dev_score", checkpoint.get("best_validation_score")
        )
    return checkpoint.get("best_validation_score")

## 8. M1 — optimized BiLSTM joint model

- **Load:** Restore the original best M1 checkpoint.
- **Train:** Initialize M1 using the original base phase, then fine-tune it with the existing learning rate and patience.

The base phase is part of M1's training procedure; it is not a fourth ensemble network.


In [14]:
# M1: shared BiLSTM encoder with attention + BIOES-CRF slot head.
seed_everything(SEED)
model1 = M1Joint()

if MODE["M1"] == "load":
    model1, ck1 = load_joint("M1", model1, LOAD_PATH["M1"])
else:
    if M1_INIT_MODE == "load":
        initial_checkpoint = torch.load(
            LOAD_PATH["M1_init"], map_location="cpu", weights_only=True
        )

        # These mappings must match the model that produced the base weights.
        if initial_checkpoint["word_to_id"] != word_to_id:
            raise ValueError("M1 base checkpoint word vocabulary mismatch")
        if initial_checkpoint["intent_to_id"] != intent_to_id:
            raise ValueError("M1 base checkpoint intent mapping mismatch")
        if initial_checkpoint["label_to_id"] != bioes_label_to_id:
            raise ValueError("M1 base checkpoint BIOES mapping mismatch")

        model1.load_state_dict(initial_checkpoint["weights"], strict=True)
        print("Restored M1 base initialization:", LOAD_PATH["M1_init"])

    else:
        # Initial training starts with permitted pretrained GloVe word vectors.
        with torch.no_grad():
            model1.word_embedding.weight.copy_(m1_initial_matrix)

        model1, initial_checkpoint = train_joint(
            name="M1_init",
            model=model1,
            train_batches=m1_train_loader,
            dev_batches=m1_dev_loader,
            lr=1e-3,
            max_epochs=M1_INIT_EPOCHS,
            patience=M1_INIT_EPOCHS,
            checkpoint_file=OUTPUT_DIR / "M1_initialization_best.pt",
        )

    # Fine-tune using the original optimized M1 settings.
    model1, ck1 = train_joint(
        name="M1",
        model=model1,
        train_batches=m1_train_loader,
        dev_batches=m1_dev_loader,
        lr=M1_LR,
        max_epochs=M1_EPOCHS,
        patience=M1_PATIENCE,
        checkpoint_file=OUTPUT_DIR / "M1_best.pt",
    )

print("M1 best epoch:", ck1["best_epoch"])
print("M1 parameters:", f"{sum(p.numel() for p in model1.parameters()):,}")


[2026-11-11T17:17:23+08:00] START M1_init training
[2026-11-11T17:17:27+08:00] START M1_init epoch 01
[2026-11-11T17:20:28+08:00] END M1_init epoch 01 | elapsed 0:03:01
M1_init epoch 01 | train loss 8.8894 | dev loss 4.4259 | intent 52.68% | slot F1 58.73% | combined 55.71
  Saved new best checkpoint: final_M1_M2_M3_outputs_corrected\M1_initialization_best.pt
[2026-11-11T17:20:28+08:00] START M1_init epoch 02
[2026-11-11T17:23:30+08:00] END M1_init epoch 02 | elapsed 0:03:01
M1_init epoch 02 | train loss 3.4216 | dev loss 2.5491 | intent 75.11% | slot F1 72.87% | combined 73.99
  Saved new best checkpoint: final_M1_M2_M3_outputs_corrected\M1_initialization_best.pt
[2026-11-11T17:23:30+08:00] START M1_init epoch 03
[2026-11-11T17:26:32+08:00] END M1_init epoch 03 | elapsed 0:03:03
M1_init epoch 03 | train loss 2.3000 | dev loss 2.0651 | intent 80.28% | slot F1 77.13% | combined 78.70
  Saved new best checkpoint: final_M1_M2_M3_outputs_corrected\M1_initialization_best.pt
[2026-11-11T17:2

## 9. M2 — residual Character CNN + joint BiLSTM + CRF

M2 adds character embeddings, a convolution, and a 300D projection to M1. Training **starts with the compatible M1 weights**, while the new character layers start fresh. Loading uses the original CharacterCNN checkpoint.


In [15]:
# M2: joint model with residual character features.
seed_everything(SEED)
model2 = M2CharacterJoint()

if MODE["M2"] == "load":
    model2, ck2 = load_joint("M2", model2, LOAD_PATH["M2"])
else:
    # Transfer all compatible M1 parameters to M2.
    transfer_status = model2.load_state_dict(model1.state_dict(), strict=False)

    expected_new_character_layers = {
        "character_embedding.weight",
        "character_convolution.weight",
        "character_convolution.bias",
        "character_projection.weight",
        "character_projection.bias",
    }
    if transfer_status.unexpected_keys:
        raise ValueError(f"Unexpected M1→M2 weights: {transfer_status.unexpected_keys}")
    if set(transfer_status.missing_keys) != expected_new_character_layers:
        raise ValueError(f"Unexpected missing M2 layers: {transfer_status.missing_keys}")

    model2, ck2 = train_joint(
        name="M2",
        model=model2,
        train_batches=train_loader,
        dev_batches=dev_loader,
        lr=M2_LR,
        max_epochs=M2_EPOCHS,
        patience=M2_PATIENCE,
        checkpoint_file=OUTPUT_DIR / "M2_best.pt",
    )

print("M2 best epoch:", ck2["best_epoch"])
print("M2 parameters:", f"{sum(p.numel() for p in model2.parameters()):,}")


[2026-11-11T18:16:14+08:00] START M2 training
[2026-11-11T18:16:14+08:00] START M2 epoch 01
[2026-11-11T18:19:39+08:00] END M2 epoch 01 | elapsed 0:03:25
M2 epoch 01 | train loss 0.5013 | dev loss 1.7351 | intent 85.34% | slot F1 80.42% | combined 82.88
  Saved new best checkpoint: final_M1_M2_M3_outputs_corrected\M2_best.pt
[2026-11-11T18:19:39+08:00] START M2 epoch 02
[2026-11-11T18:22:56+08:00] END M2 epoch 02 | elapsed 0:03:16
M2 epoch 02 | train loss 0.4747 | dev loss 1.7285 | intent 85.29% | slot F1 80.42% | combined 82.86
[2026-11-11T18:22:56+08:00] START M2 epoch 03
[2026-11-11T18:26:11+08:00] END M2 epoch 03 | elapsed 0:03:16
M2 epoch 03 | train loss 0.4475 | dev loss 1.7556 | intent 85.54% | slot F1 80.04% | combined 82.79
[2026-11-11T18:26:11+08:00] START M2 epoch 04
[2026-11-11T18:29:27+08:00] END M2 epoch 04 | elapsed 0:03:15
M2 epoch 04 | train loss 0.4306 | dev loss 1.7482 | intent 85.54% | slot F1 80.42% | combined 82.98
  Saved new best checkpoint: final_M1_M2_M3_outpu

## 10. M3 — frozen GloVe + trainable BPE + joint BiLSTM-CRF

M3 is trained **independently**, with its own BPE embeddings and a two-layer BiLSTM. Loading restores the saved network and its matching tokenizer; there is no M1/M2 warm start.


In [16]:
# M3: independently trained joint model using GloVe and BPE.
seed_everything(SEED)
model3 = M3Joint(m3_glove_matrix)

if MODE["M3"] == "load":
    model3, ck3 = load_joint("M3", model3, LOAD_PATH["M3"])
else:
    model3, ck3 = train_joint(
        name="M3",
        model=model3,
        train_batches=train_loader,
        dev_batches=dev_loader,
        lr=M3_LR,
        max_epochs=M3_EPOCHS,
        patience=M3_PATIENCE,
        checkpoint_file=OUTPUT_DIR / "M3_best.pt",
    )

print("M3 best epoch:", ck3["best_epoch"])
print("M3 parameters:", f"{sum(p.numel() for p in model3.parameters()):,}")


[2026-11-11T18:52:22+08:00] START M3 training
[2026-11-11T18:52:22+08:00] START M3 epoch 01
[2026-11-11T18:55:45+08:00] END M3 epoch 01 | elapsed 0:03:23
M3 epoch 01 | train loss 9.5879 | dev loss 6.0748 | intent 30.20% | slot F1 41.88% | combined 36.04
  Saved new best checkpoint: final_M1_M2_M3_outputs_corrected\M3_best.pt
[2026-11-11T18:55:45+08:00] START M3 epoch 02
[2026-11-11T18:59:06+08:00] END M3 epoch 02 | elapsed 0:03:21
M3 epoch 02 | train loss 4.9538 | dev loss 3.5392 | intent 59.47% | slot F1 61.77% | combined 60.62
  Saved new best checkpoint: final_M1_M2_M3_outputs_corrected\M3_best.pt
[2026-11-11T18:59:06+08:00] START M3 epoch 03
[2026-11-11T19:02:50+08:00] END M3 epoch 03 | elapsed 0:03:44
M3 epoch 03 | train loss 3.2633 | dev loss 2.5935 | intent 73.83% | slot F1 69.93% | combined 71.88
  Saved new best checkpoint: final_M1_M2_M3_outputs_corrected\M3_best.pt
[2026-11-11T19:02:50+08:00] START M3 epoch 04
[2026-11-11T19:07:06+08:00] END M3 epoch 04 | elapsed 0:04:16
M3 

## 11. Validate each model and enforce the **combined** parameter limit

Each model is evaluated separately on dev. For loaded checkpoints, the notebook checks that the newly computed score matches the saved best score closely. It then sums **all three networks' parameters**, including embeddings and CRF layers, rather than checking each model alone.


In [17]:
# Keep completed networks and their checkpoint metadata together.
models_joint = {
    "M1": model1,
    "M2": model2,
    "M3": model3,
}
checkpoints = {
    "M1": ck1,
    "M2": ck2,
    "M3": ck3,
}

standalone = {}
for model_name, model in models_joint.items():
    validation_result = evaluate_model(model, dev_loader, model_name)
    saved_best_score = checkpoint_best_score(model_name, checkpoints[model_name])

    print(f"{model_name} validation metrics: {validation_result}")
    print(f"  Saved best combined score: {saved_best_score:.4f}")

    if abs(validation_result["combined_score"] - saved_best_score) >= 0.06:
        raise ValueError(
            f"{model_name}: saved and computed validation scores disagree. "
            "Check tokenizer and vocabulary mappings."
        )

    standalone[model_name] = validation_result

# All three CRFs must use identical BIOES transition constraints.
for buffer_name in ("legal_transition", "legal_start", "legal_end"):
    m1_constraint = getattr(model1.crf, buffer_name)
    m2_constraint = getattr(model2.crf, buffer_name)
    m3_constraint = getattr(model3.crf, buffer_name)

    if not torch.equal(m1_constraint, m2_constraint):
        raise ValueError(f"M1/M2 CRF constraints differ: {buffer_name}")
    if not torch.equal(m1_constraint, m3_constraint):
        raise ValueError(f"M1/M3 CRF constraints differ: {buffer_name}")

parameter_counts = {}
trainable_parameter_counts = {}
frozen_parameter_counts = {}

for model_name, model in models_joint.items():
    all_parameters = sum(parameter.numel() for parameter in model.parameters())
    trainable_parameters = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    parameter_counts[model_name] = all_parameters
    trainable_parameter_counts[model_name] = trainable_parameters
    frozen_parameter_counts[model_name] = all_parameters - trainable_parameters

total_parameters = sum(parameter_counts.values())
total_trainable_parameters = sum(trainable_parameter_counts.values())

print("Parameter count by model (frozen embeddings count toward the budget):")
for model_name in models_joint:
    print(
        f"  {model_name}: "
        f"total={parameter_counts[model_name]:,}, "
        f"trainable={trainable_parameter_counts[model_name]:,}, "
        f"frozen={frozen_parameter_counts[model_name]:,}"
    )
print(f"Ensemble total: {total_parameters:,} / {PARAMETER_LIMIT:,}")
print(f"Ensemble trainable parameters: {total_trainable_parameters:,}")

if total_parameters > PARAMETER_LIMIT:
    raise ValueError("Ensemble exceeds the 20-million parameter limit")


M1 validation metrics: {'intent_accuracy': 85.98130841121495, 'slot_f1': 80.75282308657465, 'combined_score': 83.3670657488948}
  Saved best combined score: 83.3671
M2 validation metrics: {'intent_accuracy': 85.78455484505656, 'slot_f1': 80.40050062578223, 'combined_score': 83.0925277354194}
  Saved best combined score: 83.0925
M3 validation metrics: {'intent_accuracy': 86.17806197737335, 'slot_f1': 80.07978060334081, 'combined_score': 83.12892129035708}
  Saved best combined score: 83.1289
Parameter count by model (frozen embeddings count toward the budget):
  M1: total=2,153,277, trainable=2,153,277, frozen=0
  M2: total=2,192,809, trainable=2,192,809, frozen=0
  M3: total=5,022,737, trainable=3,431,837, frozen=1,590,900
Ensemble total: 9,368,823 / 20,000,000
Ensemble trainable parameters: 7,777,923


## 12. Equal-weight ensemble — intent probabilities + BIOES CRF scores

For **intent classification**, average the softmax probabilities from M1, M2, and M3. For **slot filling**, average each model's token emission scores and its learned CRF transition/start/end scores, then run Viterbi once on the combined potentials. The weights are fixed at one-third each.


In [18]:
@torch.inference_mode()
def fused_viterbi(emissions, lengths, transitions, starts, ends):
    """Find the highest-scoring BIOES tag path under fused CRF scores.

    emissions:   [batch, max_words, num_tags]
    transitions: [num_tags, num_tags]
    starts/ends: [num_tags]
    """
    batch_size, maximum_words, tag_count = emissions.shape
    lengths = lengths.to(emissions.device)

    # First token: start score + its emission score for each possible tag.
    best_scores = starts.unsqueeze(0) + emissions[:, 0, :]
    back_pointers = []

    for token_index in range(1, maximum_words):
        candidate_scores = (
            best_scores.unsqueeze(2) + transitions.unsqueeze(0)
        )
        best_previous_scores, previous_tag_ids = candidate_scores.max(dim=1)
        updated_scores = best_previous_scores + emissions[:, token_index, :]

        # Don't update a sentence after it reaches its real token length.
        valid_sentence = (lengths > token_index).unsqueeze(1)
        best_scores = torch.where(valid_sentence, updated_scores, best_scores)
        back_pointers.append(previous_tag_ids)

    final_tag_ids = (best_scores + ends.unsqueeze(0)).argmax(dim=1)
    best_paths = []

    for example_index in range(batch_size):
        current_path = [int(final_tag_ids[example_index])]
        sentence_length = int(lengths[example_index])

        for token_index in range(sentence_length - 1, 0, -1):
            preceding_tag_id = back_pointers[token_index - 1][
                example_index, current_path[-1]
            ]
            current_path.append(int(preceding_tag_id))

        best_paths.append(list(reversed(current_path)))

    return best_paths


@torch.inference_mode()
def evaluate_ensemble(loader):
    """Evaluate the three-model ensemble on dev or the frozen test split."""
    for model in models_joint.values():
        model.eval()

    # The CRF transition/start/end parameters are averaged across models.
    model_crf_scores = {
        model_name: model.crf.constrained_scores()
        for model_name, model in models_joint.items()
    }
    fused_crf_scores = [
        sum(
            FUSION_WEIGHTS[model_name] * model_crf_scores[model_name][part_index]
            for model_name in FUSION_WEIGHTS
        )
        for part_index in range(3)
    ]

    correct_intents = 0
    total_examples = 0
    true_positive = 0
    false_positive = 0
    false_negative = 0

    for batch in loader:
        # Every model sees the same original word boundaries.
        predictions = {
            model_name: model_outputs(model, batch, model_name)
            for model_name, model in models_joint.items()
        }

        emission_shapes = [slot_scores.shape for _, slot_scores in predictions.values()]
        if not all(shape == emission_shapes[0] for shape in emission_shapes):
            raise ValueError("The three models returned misaligned BIOES emissions")

        # Average intent class probabilities, then select the largest class.
        averaged_intent_probabilities = sum(
            FUSION_WEIGHTS[model_name] * intent_logits.softmax(dim=-1)
            for model_name, (intent_logits, _) in predictions.items()
        )
        predicted_intent_ids = averaged_intent_probabilities.argmax(dim=1).cpu()
        correct_intents += int(
            (predicted_intent_ids == batch["intent_ids"]).sum().item()
        )
        total_examples += len(batch["intent_ids"])

        # Average token-level slot emission scores before CRF decoding.
        averaged_slot_emissions = sum(
            FUSION_WEIGHTS[model_name] * slot_emissions
            for model_name, (_, slot_emissions) in predictions.items()
        )
        predicted_slot_paths = fused_viterbi(
            averaged_slot_emissions,
            batch["lengths"],
            *fused_crf_scores,
        )

        for example_index, predicted_tag_ids in enumerate(predicted_slot_paths):
            predicted_tag_names = [TAG_NAMES[tag_id] for tag_id in predicted_tag_ids]
            predicted_slots = decode_tagged_spans(
                predicted_tag_names,
                batch["token_offsets"][example_index],
                batch["utterances"][example_index],
                label_scheme="BIOES",
            )
            correct, extra, missing = span_counts(
                predicted_slots,
                batch["gold_slot_pairs"][example_index],
            )
            true_positive += correct
            false_positive += extra
            false_negative += missing

    intent_accuracy = 100.0 * correct_intents / total_examples
    slot_f1 = exact_micro_f1(true_positive, false_positive, false_negative)

    return {
        "intent_accuracy": intent_accuracy,
        "slot_f1": slot_f1,
        "combined_score": (intent_accuracy + slot_f1) / 2.0,
        "parameters": total_parameters,
        "examples": total_examples,
    }


ensemble_clock = start_clock("M1 + M2 + M3 ensemble validation")
validation_metrics = evaluate_ensemble(dev_loader)
stop_clock(
    "M1 + M2 + M3 ensemble validation",
    ensemble_clock,
    action="evaluation",
)
print("Ensemble validation results:", validation_metrics)


[2026-11-11T20:04:49+08:00] START M1 + M2 + M3 ensemble validation
[2026-11-11T20:04:54+08:00] END M1 + M2 + M3 ensemble validation | elapsed 0:00:06
Ensemble validation results: {'intent_accuracy': 86.91588785046729, 'slot_f1': 82.02303455182773, 'combined_score': 84.46946120114751, 'parameters': 9368823, 'examples': 2033}


## 13. Export the model bundle, dev results, and training-time CSV

Store all three trained/loaded state dictionaries, their shared label IDs, tokenizer, ensemble weights, parameter count, and validation metrics in a single reuse bundle. Saving the bundle does **not** create another neural network.


In [19]:
# Save CPU copies so the bundle can be read without a GPU.
model_weight_dict = {}
for model_name, model in models_joint.items():
    model_weight_dict[model_name] = {
        tensor_name: tensor.detach().cpu().clone()
        for tensor_name, tensor in model.state_dict().items()
    }

bundle = {
    "model_names": ["M1", "M2", "M3"],
    "weights": model_weight_dict,
    "bpe_tokenizer_json": bpe.to_str(),
    "word_to_id": word_to_id,
    "word_index": word_index,
    "char_to_id": char_to_id,
    "intent_to_id": intent_to_id,
    "bioes_label_to_id": bioes_label_to_id,
    "fusion_weights": dict(FUSION_WEIGHTS),
    "parameter_count": total_parameters,
    "trainable_parameter_count": total_trainable_parameters,
    "parameter_counts_by_model": parameter_counts,
    "trainable_parameters_by_model": trainable_parameter_counts,
    "m3_configuration": {
        "hidden_size": M3_HIDDEN_SIZE,
        "layers": M3_LAYERS,
        "dropout": M3_DROPOUT,
        "subword_dim": SUBWORD_DIM,
    },
    "validation_metrics": validation_metrics,
    "modes": dict(MODE),
    "saved_at_sgt": now_sgt(),
}

bundle_file = OUTPUT_DIR / "M1_M2_M3_ensemble_bundle.pt"
torch.save(bundle, bundle_file)


def restore_ensemble_from_bundle(bundle_path):
    """Rebuild all three models from a saved bundle, not in-memory weights.

    Also restores the vocabularies, tokenizer and ensemble weights needed
    by the current notebook's model classes, collate function and evaluator.
    """
    global models_joint, word_to_id, word_index, char_to_id
    global intent_to_id, bioes_label_to_id, TAG_NAMES, bpe, BPE_PAD, BPE_UNK
    global FUSION_WEIGHTS, total_parameters, total_trainable_parameters
    global M3_HIDDEN_SIZE, M3_LAYERS, M3_DROPOUT, SUBWORD_DIM

    saved_bundle = torch.load(
        bundle_path, map_location="cpu", weights_only=True
    )

    word_to_id = saved_bundle["word_to_id"]
    word_index = saved_bundle["word_index"]
    char_to_id = saved_bundle["char_to_id"]
    intent_to_id = saved_bundle["intent_to_id"]
    bioes_label_to_id = saved_bundle["bioes_label_to_id"]
    TAG_NAMES = [
        tag for tag, tag_id in sorted(
            bioes_label_to_id.items(), key=lambda item: item[1]
        )
    ]

    bpe = Tokenizer.from_str(saved_bundle["bpe_tokenizer_json"])
    BPE_PAD = bpe.token_to_id("<pad>")
    BPE_UNK = bpe.token_to_id("<unk>")
    if BPE_PAD is None or BPE_UNK is None:
        raise ValueError("Saved tokenizer is missing padding/unknown tokens")

    configuration = saved_bundle["m3_configuration"]
    M3_HIDDEN_SIZE = configuration["hidden_size"]
    M3_LAYERS = configuration["layers"]
    M3_DROPOUT = configuration["dropout"]
    SUBWORD_DIM = configuration["subword_dim"]

    FUSION_WEIGHTS = saved_bundle["fusion_weights"]
    total_parameters = saved_bundle["parameter_count"]
    total_trainable_parameters = saved_bundle["trainable_parameter_count"]

    # M3 only needs a tensor of the correct size for construction:
    # its actual frozen GloVe values come from the saved state dictionary.
    m3_embedding_shape = saved_bundle["weights"]["M3"]["word_embedding.weight"].shape
    empty_glove_matrix = torch.zeros(m3_embedding_shape, dtype=torch.float32)

    restored_models = {
        "M1": M1Joint(),
        "M2": M2CharacterJoint(),
        "M3": M3Joint(empty_glove_matrix),
    }

    restored_total = 0
    restored_trainable = 0
    for model_name, restored_model in restored_models.items():
        restored_model.load_state_dict(
            saved_bundle["weights"][model_name], strict=True
        )
        restored_model.to(device)
        restored_model.eval()
        restored_total += sum(p.numel() for p in restored_model.parameters())
        restored_trainable += sum(
            p.numel() for p in restored_model.parameters() if p.requires_grad
        )

    assert restored_total == total_parameters, "Restored parameter count differs"
    assert restored_trainable == total_trainable_parameters, (
        "Restored trainable parameter count differs"
    )

    # The ensemble evaluator uses this dictionary, not model1/model2/model3.
    models_joint = restored_models
    print(f"Reloaded M1, M2 and M3 from {bundle_path}")
    return saved_bundle


# One row per individual joint model, followed by the three-way ensemble.
score_rows = []
for model_name, metrics in standalone.items():
    score_rows.append({
        "model": model_name,
        **metrics,
        "parameters": parameter_counts[model_name],
    })
score_rows.append({"model": "Ensemble", **validation_metrics})

results_file = OUTPUT_DIR / "M1_M2_M3_dev_results.csv"
pd.DataFrame(score_rows).to_csv(results_file, index=False)

timings_file = OUTPUT_DIR / "M1_M2_M3_training_times.csv"
pd.DataFrame(TIMING_ROWS).to_csv(timings_file, index=False)

print("Saved files:")
for output_file in (bundle_file, results_file, timings_file, BPE_SAVE_PATH):
    print(" ", output_file)

print("Training/loading time by component:")
for timing in TIMING_ROWS:
    if timing["action"] in ("train", "load"):
        print(
            f"  {timing['component']}: "
            f"{timing['elapsed_seconds']:.1f}s ({timing['action']})"
        )


Saved files:
  final_M1_M2_M3_outputs_corrected\M1_M2_M3_ensemble_bundle.pt
  final_M1_M2_M3_outputs_corrected\M1_M2_M3_dev_results.csv
  final_M1_M2_M3_outputs_corrected\M1_M2_M3_training_times.csv
  final_M1_M2_M3_outputs_corrected\M3_bpe_tokenizer.json
Training/loading time by component:
  M1_init training: 2234.3s (train)
  M1 training: 1295.9s (train)
  M2 training: 2167.8s (train)
  M3 training: 4328.1s (train)


## 14. Mandatory saved-bundle reload verification + optional test

Leave `RUN_TEST_ONCE = False` during model selection. Turn it on only after fixing all model weights, preprocessing, and ensemble settings for your final evaluation.


In [21]:
# Always evaluate freshly reconstructed models from the saved bundle.
# The following call replaces models_joint with new model instances.
reload_clock = start_clock("Reload final ensemble bundle")
restored_bundle = restore_ensemble_from_bundle(bundle_file)
stop_clock("Reload final ensemble bundle", reload_clock, action="load")

# Build fresh evaluation loaders after restoring the tokenizer and vocabularies.
restored_dev_loader = make_loader(
    JointDataset(validation_records), BATCH_SIZE, shuffle=False
)

reloaded_dev_clock = start_clock("Reloaded ensemble dev evaluation")
reloaded_dev_metrics = evaluate_ensemble(restored_dev_loader)
stop_clock(
    "Reloaded ensemble dev evaluation", reloaded_dev_clock, action="evaluation"
)

# Do not assert a hard-coded 84.61: training M3 with train-only vocabulary
# changes the ensemble, so the new validation score may be different.
for metric_name in ("intent_accuracy", "slot_f1", "combined_score"):
    expected_value = restored_bundle["validation_metrics"][metric_name]
    actual_value = reloaded_dev_metrics[metric_name]
    assert abs(actual_value - expected_value) < 1e-6, (
        f"Reload verification failed for {metric_name}: "
        f"saved={expected_value}, reloaded={actual_value}"
    )

assert reloaded_dev_metrics["parameters"] == restored_bundle["parameter_count"]
assert reloaded_dev_metrics["examples"] == len(validation_records)

print("Bundle reload verification passed.")
print("Recomputed dev metrics:", reloaded_dev_metrics)

# Save the verified metrics, even when the held-out test remains disabled.
pd.DataFrame([reloaded_dev_metrics]).to_csv(
    OUTPUT_DIR / "M1_M2_M3_reloaded_dev_results.csv", index=False
)
RUN_TEST_ONCE=True
if RUN_TEST_ONCE:
    # Do not use test scores to tune architectures, preprocessing or weights.
    restored_test_loader = make_loader(
        JointDataset(test_records), BATCH_SIZE, shuffle=False
    )

    test_clock = start_clock("Reloaded M1/M2/M3 test evaluation")
    test_metrics = evaluate_ensemble(restored_test_loader)
    stop_clock(
        "Reloaded M1/M2/M3 test evaluation",
        test_clock,
        action="evaluation",
    )

    print("Held-out test metrics:", test_metrics)
    pd.DataFrame([test_metrics]).to_csv(
        OUTPUT_DIR / "M1_M2_M3_test_results.csv", index=False
    )
else:
    print("Held-out test evaluation disabled (RUN_TEST_ONCE=False)")

# Update timing CSV with the bundle reload and final evaluation timings.
pd.DataFrame(TIMING_ROWS).to_csv(
    OUTPUT_DIR / "M1_M2_M3_training_times.csv", index=False
)


[2026-11-11T20:05:55+08:00] START Reload final ensemble bundle
Reloaded M1, M2 and M3 from final_M1_M2_M3_outputs_corrected\M1_M2_M3_ensemble_bundle.pt
[2026-11-11T20:05:56+08:00] END Reload final ensemble bundle | elapsed 0:00:00
[2026-11-11T20:05:57+08:00] START Reloaded ensemble dev evaluation
[2026-11-11T20:06:03+08:00] END Reloaded ensemble dev evaluation | elapsed 0:00:06
Bundle reload verification passed.
Recomputed dev metrics: {'intent_accuracy': 86.91588785046729, 'slot_f1': 82.02303455182773, 'combined_score': 84.46946120114751, 'parameters': 9368823, 'examples': 2033}
[2026-11-11T20:06:03+08:00] START Reloaded M1/M2/M3 test evaluation
[2026-11-11T20:06:11+08:00] END Reloaded M1/M2/M3 test evaluation | elapsed 0:00:08
Held-out test metrics: {'intent_accuracy': 86.21385339609952, 'slot_f1': 79.30971843778383, 'combined_score': 82.76178591694168, 'parameters': 9368823, 'examples': 2974}
